# Final System Colab Runner

This notebook runs the final custom RAG benchmark system in Google Colab.

It is intended for GPU-based testing of:

- retrieval-only mode
- base RAG mode
- fine-tuned RAG mode
- base vs fine-tuned comparison mode

The final runnable system is located under `final_system/` in the GitHub repository.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Please use Runtime > Change runtime type > L4 GPU.")

CUDA available: True
GPU: NVIDIA L4


In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/Kutay11019/turkish-legal-rag.git"
REPO_DIR = Path("/content/turkish-legal-rag")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}

print("Repo cloned to:", REPO_DIR)
print("final_system exists:", (REPO_DIR / "final_system").exists())

Cloning into '/content/turkish-legal-rag'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 206 (delta 98), reused 133 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 779.59 KiB | 8.38 MiB/s, done.
Resolving deltas: 100% (98/98), done.
Repo cloned to: /content/turkish-legal-rag
final_system exists: True


## Using the Sample Benchmark or Your Own Benchmark

This notebook can be executed with the included sample files. However, the provided `sample_legal_document.txt` and `sample_benchmark.csv` are only small smoke-test examples. They contain only two simple test questions, so the generated scores should not be interpreted as real project performance.

If you want to test the system with your own benchmark, you can use one of the following options:

1. Open `final_system/data/custom_benchmark/sample_benchmark.csv` and replace its content with your own benchmark questions.
2. Delete the existing `sample_benchmark.csv` file and upload your own benchmark file with the same name: `sample_benchmark.csv`.

Your benchmark CSV must contain at least these columns:

```csv
question,expected_answer
```

If you also want to use your own legal documents, place them under:

```text
final_system/data/custom_documents/
```

After adding or replacing the benchmark/document files, you do not need to restart the notebook from the beginning. Continue by running the next cell after the file upload/replacement step.

In [ ]:
FINAL_SYSTEM_DIR = REPO_DIR / "final_system"

%cd {FINAL_SYSTEM_DIR}

!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

# Needed for 4-bit GPU loading and Drive download
!python -m pip install -q bitsandbytes gdown

/content/turkish-legal-rag/final_system
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 157.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pymupdf]


In [ ]:
from pathlib import Path
import subprocess
import os

ADAPTER_DRIVE_FILE_LINK = "https://drive.google.com/file/d/1VyKe4-oydf8LUs_rmnu02079tFCZkIGO/view?usp=sharing"

DOWNLOAD_DIR = Path("/content/adapter_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

adapter_zip_path = DOWNLOAD_DIR / "mistral_legal_qlora_starlar_v2_800steps_clean.zip"

print("Downloading adapter zip from Google Drive...")
print("Output path:", adapter_zip_path)

result = subprocess.run(
    [
        "gdown",
        "--fuzzy",
        ADAPTER_DRIVE_FILE_LINK,
        "-O",
        str(adapter_zip_path)
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Zip exists:", adapter_zip_path.exists())

if not adapter_zip_path.exists():
    raise FileNotFoundError("Adapter zip was not downloaded. Check the direct file sharing link.")

print("Zip size MB:", round(adapter_zip_path.stat().st_size / (1024 * 1024), 2))

Output path: /content/adapter_download/mistral_legal_qlora_starlar_v2_800steps_clean.zip
STDOUT:

STDERR:
Downloading...
From (original): https://drive.google.com/uc?id=1VyKe4-oydf8LUs_rmnu02079tFCZkIGO
From (redirected): https://drive.google.com/uc?id=1VyKe4-oydf8LUs_rmnu02079tFCZkIGO&confirm=t&uuid=b743f9d5-a8bf-4ce3-83bf-83530e713b7c
To: /content/adapter_download/mistral_legal_qlora_starlar_v2_800steps_clean.zip

  0%|          | 0.00/156M [00:00<?, ?B/s]
  4%|▍         | 6.82M/156M [00:00<00:02, 53.5MB/s]
  9%|▉         | 14.7M/156M [00:00<00:02, 64.7MB/s]
 15%|█▌        | 23.6M/156M [00:00<00:01, 68.0MB/s]
 23%|██▎       | 35.7M/156M [00:00<00:01, 85.8MB/s]
 31%|███▏      | 48.8M/156M [00:00<00:01, 94.6MB/s]
 41%|████      | 63.4M/156M [00:00<00:00, 110MB/s] 
 48%|████▊     | 75.0M/156M [00:00<00:00, 95.0MB/s]
 54%|█████▍    | 84.9M/156M [00:00<00:00, 85.2MB/s]
 61%|██████    | 94.9M/156M [00:01<00:00, 88.5MB/s]
 68%|██████▊   | 105M/156M [00:01<00:00, 90.9MB/s] 
 74%|███████▎  | 

In [ ]:
import zipfile
import shutil

ADAPTER_TARGET_DIR = FINAL_SYSTEM_DIR / "models" / "mistral_legal_qlora_starlar_v2_800steps"

if ADAPTER_TARGET_DIR.exists():
    shutil.rmtree(ADAPTER_TARGET_DIR)

ADAPTER_TARGET_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(adapter_zip_path, "r") as z:
    z.extractall(ADAPTER_TARGET_DIR)

print("Adapter extracted to:", ADAPTER_TARGET_DIR)

print("\nFiles:")
for item in sorted(ADAPTER_TARGET_DIR.iterdir()):
    print("-", item.name)

print("\nRequired file check:")
print("adapter_config.json:", (ADAPTER_TARGET_DIR / "adapter_config.json").exists())
print("adapter_model.safetensors:", (ADAPTER_TARGET_DIR / "adapter_model.safetensors").exists())

Adapter extracted to: /content/turkish-legal-rag/final_system/models/mistral_legal_qlora_starlar_v2_800steps

Files:
- README.md
- adapter_config.json
- adapter_model.safetensors
- chat_template.jinja
- tokenizer.json
- tokenizer_config.json
- training_args.bin

Required file check:
adapter_config.json: True
adapter_model.safetensors: True


In [ ]:
import yaml

CONFIG_PATH = FINAL_SYSTEM_DIR / "config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["models"]["local_finetuned_adapter"] = "models/mistral_legal_qlora_starlar_v2_800steps"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print("Updated config.yaml")
print("local_finetuned_adapter:", config["models"]["local_finetuned_adapter"])

Updated config.yaml
local_finetuned_adapter: models/mistral_legal_qlora_starlar_v2_800steps


In [ ]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
%cd {FINAL_SYSTEM_DIR}

!python run_custom_rag_benchmark.py --mode base --retrieval_only

/content/turkish-legal-rag/final_system
Turkish Legal RAG Final System
Documents directory: /content/turkish-legal-rag/final_system/data/custom_documents
Benchmark path: /content/turkish-legal-rag/final_system/data/custom_benchmark/sample_benchmark.csv
Output directory: /content/turkish-legal-rag/final_system/data/outputs
Mode: base
Retrieval only: True
Loaded documents: 1
Built chunks: 1
Benchmark rows: 2
modules.json: 100% 229/229 [00:00<00:00, 963kB/s]
config_sentence_transformers.json: 100% 122/122 [00:00<00:00, 604kB/s]
README.md: 100% 3.89k/3.89k [00:00<00:00, 2.26MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 287kB/s]
config.json: 100% 645/645 [00:00<00:00, 3.33MB/s]
model.safetensors: 100% 471M/471M [00:02<00:00, 216MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 1252.22it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
----------------

In [ ]:
import pandas as pd

base_results_path = FINAL_SYSTEM_DIR / "data" / "outputs" / "base_rag_results.csv"
summary_path = FINAL_SYSTEM_DIR / "data" / "outputs" / "base_vs_finetuned_summary.csv"

base_results_df = pd.read_csv(base_results_path)
summary_df = pd.read_csv(summary_path)

display(summary_df)
display(base_results_df[["question", "expected_answer", "top1_file_name", "top1_context"]])

,mode,sample_count,mean_token_f1,mean_text_similarity,source_citation_rate
0,base,2,0.0,0.0,0.0


,question,expected_answer,top1_file_name,top1_context
0,Anayasa 10. madde neyi düzenler?,"Anayasa 10. madde, herkesin kanun önünde eşit ...",sample_legal_document.txt,Türkiye Cumhuriyeti Anayasası Madde 10 – Herke...
1,Bilgi edinme hakkı ihlal edilirse ne yapılabilir?,Bilgi edinme hakkının ihlali durumunda ilgili ...,sample_legal_document.txt,Türkiye Cumhuriyeti Anayasası Madde 10 – Herke...


In [ ]:
%cd {FINAL_SYSTEM_DIR}

!python run_custom_rag_benchmark.py --mode base

/content/turkish-legal-rag/final_system
Turkish Legal RAG Final System
Documents directory: /content/turkish-legal-rag/final_system/data/custom_documents
Benchmark path: /content/turkish-legal-rag/final_system/data/custom_benchmark/sample_benchmark.csv
Output directory: /content/turkish-legal-rag/final_system/data/outputs
Mode: base
Retrieval only: False
Loaded documents: 1
Built chunks: 1
Benchmark rows: 2
Loading weights: 100% 199/199 [00:00<00:00, 969.47it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100% 1/1 [00:00<00:00,  5.60it/s]
config.json: 100% 596/596 [00:00<00:00, 2.88MB/s]
tokenizer_config.json: 100% 2.10k/2.10k [00:00<00:00, 5.13MB/

In [ ]:
base_results_df = pd.read_csv(FINAL_SYSTEM_DIR / "data" / "outputs" / "base_rag_results.csv")
summary_df = pd.read_csv(FINAL_SYSTEM_DIR / "data" / "outputs" / "base_vs_finetuned_summary.csv")

display(summary_df)
display(base_results_df[["question", "expected_answer", "generated_answer", "top1_context"]])

,mode,sample_count,mean_token_f1,mean_text_similarity,source_citation_rate
0,base,2,0.210764,0.35705,0.0


,question,expected_answer,generated_answer,top1_context
0,Anayasa 10. madde neyi düzenler?,"Anayasa 10. madde, herkesin kanun önünde eşit ...","Anayasa 10. madde, Türkiye Cumhuriyeti Anayasa...",Türkiye Cumhuriyeti Anayasası Madde 10 – Herke...
1,Bilgi edinme hakkı ihlal edilirse ne yapılabilir?,Bilgi edinme hakkının ihlali durumunda ilgili ...,Bilgi edinme hakkı ihlal edilmese değerlendiri...,Türkiye Cumhuriyeti Anayasası Madde 10 – Herke...


In [ ]:
%cd {FINAL_SYSTEM_DIR}

!python run_custom_rag_benchmark.py --mode finetuned

/content/turkish-legal-rag/final_system
Turkish Legal RAG Final System
Documents directory: /content/turkish-legal-rag/final_system/data/custom_documents
Benchmark path: /content/turkish-legal-rag/final_system/data/custom_benchmark/sample_benchmark.csv
Output directory: /content/turkish-legal-rag/final_system/data/outputs
Mode: finetuned
Retrieval only: False
Loaded documents: 1
Built chunks: 1
Benchmark rows: 2
Loading weights: 100% 199/199 [00:00<00:00, 1101.20it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100% 1/1 [00:00<00:00,  5.51it/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 4/291 [00:00<00:24, 11.50it/s, Ma

In [ ]:
finetuned_results_df = pd.read_csv(FINAL_SYSTEM_DIR / "data" / "outputs" / "finetuned_rag_results.csv")
summary_df = pd.read_csv(FINAL_SYSTEM_DIR / "data" / "outputs" / "base_vs_finetuned_summary.csv")

display(summary_df)
display(finetuned_results_df[["question", "expected_answer", "generated_answer", "top1_context"]])

,mode,sample_count,mean_token_f1,mean_text_similarity,source_citation_rate
0,finetuned,2,0.102728,0.194487,0.5


,question,expected_answer,generated_answer,top1_context
0,Anayasa 10. madde neyi düzenler?,"Anayasa 10. madde, herkesin kanun önünde eşit ...",Anayasa 10. madde şu düzenler yapar: Türkiye C...,Türkiye Cumhuriyeti Anayasası Madde 10 – Herke...
1,Bilgi edinme hakkı ihlal edilirse ne yapılabilir?,Bilgi edinme hakkının ihlali durumunda ilgili ...,Bu açıklama şu kaynak bilgilerine dayandırılma...,Türkiye Cumhuriyeti Anayasası Madde 10 – Herke...


In [ ]:
%cd {FINAL_SYSTEM_DIR}

!python run_custom_rag_benchmark.py --mode both

/content/turkish-legal-rag/final_system
Turkish Legal RAG Final System
Documents directory: /content/turkish-legal-rag/final_system/data/custom_documents
Benchmark path: /content/turkish-legal-rag/final_system/data/custom_benchmark/sample_benchmark.csv
Output directory: /content/turkish-legal-rag/final_system/data/outputs
Mode: both
Retrieval only: False
Loaded documents: 1
Built chunks: 1
Benchmark rows: 2
Loading weights: 100% 199/199 [00:00<00:00, 1053.33it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100% 1/1 [00:00<00:00,  5.53it/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 4/291 [00:00<00:17, 16.39it/s, Materia

In [ ]:
base_results_df = pd.read_csv(FINAL_SYSTEM_DIR / "data" / "outputs" / "base_rag_results.csv")
finetuned_results_df = pd.read_csv(FINAL_SYSTEM_DIR / "data" / "outputs" / "finetuned_rag_results.csv")
summary_df = pd.read_csv(FINAL_SYSTEM_DIR / "data" / "outputs" / "base_vs_finetuned_summary.csv")

display(summary_df)

comparison_df = pd.DataFrame({
    "question": base_results_df["question"],
    "expected_answer": base_results_df["expected_answer"],
    "base_answer": base_results_df["generated_answer"],
    "finetuned_answer": finetuned_results_df["generated_answer"],
    "base_token_f1": base_results_df["token_f1"],
    "finetuned_token_f1": finetuned_results_df["token_f1"],
    "base_similarity": base_results_df["text_similarity"],
    "finetuned_similarity": finetuned_results_df["text_similarity"],
})

display(comparison_df)

,mode,sample_count,mean_token_f1,mean_text_similarity,source_citation_rate
0,base,2,0.210764,0.357050,0.0
1,finetuned,2,0.102728,0.194487,0.5


,question,expected_answer,base_answer,finetuned_answer,base_token_f1,finetuned_token_f1,base_similarity,finetuned_similarity
0,Anayasa 10. madde neyi düzenler?,"Anayasa 10. madde, herkesin kanun önünde eşit ...","Anayasa 10. madde, Türkiye Cumhuriyeti Anayasa...",Anayasa 10. madde şu düzenler yapar: Türkiye C...,0.179104,0.176471,0.226496,0.217659
1,Bilgi edinme hakkı ihlal edilirse ne yapılabilir?,Bilgi edinme hakkının ihlali durumunda ilgili ...,Bilgi edinme hakkı ihlal edilmese değerlendiri...,Bu açıklama şu kaynak bilgilerine dayandırılma...,0.242424,0.028986,0.487603,0.171315


In [ ]:
from pathlib import Path

output_files = [
    FINAL_SYSTEM_DIR / "data" / "outputs" / "base_rag_results.csv",
    FINAL_SYSTEM_DIR / "data" / "outputs" / "finetuned_rag_results.csv",
    FINAL_SYSTEM_DIR / "data" / "outputs" / "base_vs_finetuned_summary.csv",
]

print("FINAL OUTPUT FILE CHECK")
print("=" * 80)

for path in output_files:
    print(path)
    print("Exists:", path.exists())
    if path.exists():
        print("Size KB:", round(path.stat().st_size / 1024, 2))
    print("-" * 80)

FINAL OUTPUT FILE CHECK
/content/turkish-legal-rag/final_system/data/outputs/base_rag_results.csv
Exists: True
Size KB: 3.32
--------------------------------------------------------------------------------
/content/turkish-legal-rag/final_system/data/outputs/finetuned_rag_results.csv
Exists: True
Size KB: 3.64
--------------------------------------------------------------------------------
/content/turkish-legal-rag/final_system/data/outputs/base_vs_finetuned_summary.csv
Exists: True
Size KB: 0.18
--------------------------------------------------------------------------------


In [ ]:
import pandas as pd

base_df = pd.read_csv("/content/turkish-legal-rag/final_system/data/outputs/base_rag_results.csv")
ft_df = pd.read_csv("/content/turkish-legal-rag/final_system/data/outputs/finetuned_rag_results.csv")

comparison_df = pd.DataFrame({
    "question": base_df["question"],
    "expected_answer": base_df["expected_answer"],
    "base_answer": base_df["generated_answer"],
    "finetuned_answer": ft_df["generated_answer"],
    "base_token_f1": base_df["token_f1"],
    "finetuned_token_f1": ft_df["token_f1"],
    "base_similarity": base_df["text_similarity"],
    "finetuned_similarity": ft_df["text_similarity"],
})

display(comparison_df)

,question,expected_answer,base_answer,finetuned_answer,base_token_f1,finetuned_token_f1,base_similarity,finetuned_similarity
0,Anayasa 10. madde neyi düzenler?,"Anayasa 10. madde, herkesin kanun önünde eşit ...","Anayasa 10. madde, Türkiye Cumhuriyeti Anayasa...",Anayasa 10. madde şu düzenler yapar: Türkiye C...,0.179104,0.176471,0.226496,0.217659
1,Bilgi edinme hakkı ihlal edilirse ne yapılabilir?,Bilgi edinme hakkının ihlali durumunda ilgili ...,Bilgi edinme hakkı ihlal edilmese değerlendiri...,Bu açıklama şu kaynak bilgilerine dayandırılma...,0.242424,0.028986,0.487603,0.171315
